# 00 – Download vet clinics from OpenStreetMap with OSMNX

This notebook fetches all vet clinics in Berlin from OpenStreetMap using OSMNX,
normalises geometry to point centroids, and exports a snapshot that can be used
by downstream notebooks.

Main outputs:

- `sources/osmnx_berlin_vet_clinics_latest.geojson`  (GeoJSON with geometry)
- `cache/osmnx_berlin_vet_clinics_latest.csv`        (attributes + lat/lon, no geometry)

We also:

- reset the OSMNX MultiIndex so that `element` and `id` become columns,
- construct a `source_osm_id` field (`element/id`) for provenance,
- keep address, contact, and opening_hours attributes that are relevant for the MVP.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if (ROOT / "veterinary_clinics" / "sources").exists():
    os.chdir(ROOT / "veterinary_clinics")
    print("Changed CWD to:", Path.cwd())
else:
    print("Current CWD:", ROOT)
    print("Assuming this notebook already runs inside `veterinary_clinics`.")

Current CWD: /Users/jorge/Projects/layered-populate-data-pool-da/veterinary_clinics
Assuming this notebook already runs inside `veterinary_clinics`.


In [2]:
import osmnx as ox
import geopandas as gpd

ox.settings.log_console = True
ox.settings.use_cache = True

## 1. Fetch vet clinics from OpenStreetMap (OSMNX)

We query OSM for all features with `amenity = veterinary` within
`place_name = "Berlin, Germany"`.

OSMNX returns a GeoDataFrame with a MultiIndex `(element, id)`.  
We reset the index so that:

- `element` becomes a column (`node`, `way`, `relation`)
- `id` becomes the numeric OSM identifier (to be used as primary key later)

In [3]:
place_name = "Berlin, Germany"
tags = {"amenity": "veterinary"}

gdf_osm = ox.features.features_from_place(place_name, tags=tags)
print(f"Number of OSM features fetched: {len(gdf_osm)}")

# Turn MultiIndex (element, id) into regular columns
gdf_osm = gdf_osm.reset_index()  # -> columns 'element' and 'id'
gdf_osm["id"] = gdf_osm["id"].astype("Int64")

# Combined identifier for provenance/debugging
gdf_osm["source_osm_id"] = (
    gdf_osm["element"].astype(str) + "/" + gdf_osm["id"].astype(str)
)

gdf_osm.head()

Number of OSM features fetched: 175


,element,id,geometry,addr:city,addr:housenumber,addr:postcode,addr:street,amenity,name,note,...,heritage:operator,lda:criteria,old_name,ref:lda,wikidata,wikimedia_commons,roof:levels,historic,source:shape,source_osm_id
0,node,268917040,POINT (13.40523 52.49568),Berlin,69,10961,Baerwaldstraße,veterinary,Tierarztpraxis am Urban,Städt. Kita Baerwaldstr. 69,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,node/268917040
1,node,299795048,POINT (13.47955 52.60629),Berlin,67,13125,Straße 48,veterinary,Dr. med. vet. Elke Hartwig,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,node/299795048
2,node,347294456,POINT (13.32013 52.42972),Berlin,36,12207,Königsberger Straße,veterinary,Tierarztpraxis Dr. Bernhard Sörensen,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,node/347294456
3,node,394867279,POINT (13.27057 52.5352),NaN,NaN,NaN,NaN,veterinary,Tierarztpraxis Jeanette Koepsel,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,node/394867279
4,node,411550894,POINT (13.58964 52.50951),Berlin,19,12621,Planitzstraße,veterinary,Kleintierarztpraxis Berlin Kaulsdorf,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,node/411550894


## 2. Normalise geometry and add latitude / longitude

We ensure the CRS is WGS 84 (EPSG:4326), project to a metric CRS for accurate
centroids, compute centroids, and then convert back to WGS 84 to get `lat` and `lon`.

In [4]:
# Ensure CRS is set to WGS84
if gdf_osm.crs is None:
    gdf_osm = gdf_osm.set_crs(epsg=4326)

# Project to metric CRS for centroid computation
gdf_proj = gdf_osm.to_crs(epsg=3857)
centroids_proj = gdf_proj.geometry.centroid

# Back to WGS84
centroids_wgs84 = centroids_proj.to_crs(epsg=4326)

# Overwrite geometry with centroid in WGS84 and derive lat/lon
gdf_osm["geometry"] = centroids_wgs84
gdf_osm["lat"] = gdf_osm.geometry.y
gdf_osm["lon"] = gdf_osm.geometry.x

gdf_osm[["element", "id", "source_osm_id", "name", "addr:street", "addr:housenumber", "lat", "lon"]].head()

,element,id,source_osm_id,name,addr:street,addr:housenumber,lat,lon
0,node,268917040,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,52.495684,13.405233
1,node,299795048,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,52.606286,13.479555
2,node,347294456,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,52.429722,13.320133
3,node,394867279,node/394867279,Tierarztpraxis Jeanette Koepsel,NaN,NaN,52.535199,13.270573
4,node,411550894,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,52.509511,13.589635


## 3. Select relevant columns for downstream processing

We keep OSM identifiers, key attributes, and geometry.  
These will be enriched with district and neighborhood information in the next notebook.

In [5]:
cols_keep = [
    "element",
    "id",
    "source_osm_id",
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "phone",
    "contact:phone",
    "email",
    "contact:email",
    "website",
    "contact:website",
    "opening_hours",
    "operator",
    "wheelchair",
    "wheelchair:description",
    "emergency",
    "lat",
    "lon",
]

cols_export = ["geometry"] + [c for c in cols_keep if c in gdf_osm.columns]

gdf_osm_subset = gpd.GeoDataFrame(
    gdf_osm[cols_export].copy(),
    geometry="geometry",
    crs=gdf_osm.crs,
)

gdf_osm_subset.head()

,geometry,element,id,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,...,contact:email,website,contact:website,opening_hours,operator,wheelchair,wheelchair:description,emergency,lat,lon
0,POINT (13.40523 52.49568),node,268917040,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961,Berlin,NaN,...,NaN,NaN,NaN,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",NaN,no,NaN,NaN,52.495684,13.405233
1,POINT (13.47955 52.60629),node,299795048,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125,Berlin,+49 30 9437820,...,NaN,http://www.tierarztpraxis-hartwig.de/,NaN,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",NaN,limited,NaN,NaN,52.606286,13.479555
2,POINT (13.32013 52.42972),node,347294456,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207,Berlin,+49 30 7738321,...,NaN,https://www.tierarztpraxis-soerensen.de/,NaN,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",NaN,yes,NaN,NaN,52.429722,13.320133
3,POINT (13.27057 52.5352),node,394867279,node/394867279,Tierarztpraxis Jeanette Koepsel,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.535199,13.270573
4,POINT (13.58964 52.50951),node,411550894,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621,Berlin,+49 30 53018585,...,NaN,https://www.tierarzt-kaulsdorf.de/,NaN,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Dr. Berit Miels;Dr. Mathias Kochert,NaN,NaN,NaN,52.509511,13.589635


## 4. Export OSM snapshot (GeoJSON + CSV)

- GeoJSON keeps the geometry for spatial joins.
- CSV provides a flat snapshot with attributes + lat/lon (no geometry) for lightweight inspection.

In [6]:
from pathlib import Path

output_geojson = Path("sources/osmnx_berlin_vet_clinics_latest.geojson")
output_csv = Path("cache/osmnx_berlin_vet_clinics_latest.csv")

output_geojson.parent.mkdir(parents=True, exist_ok=True)
output_csv.parent.mkdir(parents=True, exist_ok=True)

# GeoJSON with geometry
gdf_osm_subset.to_file(output_geojson, driver="GeoJSON")

# CSV without geometry, attributes only
gdf_osm_subset.drop(columns="geometry").to_csv(output_csv, index=False)

print("Exported:")
print(" -", output_geojson)
print(" -", output_csv)
print("Shape:", gdf_osm_subset.shape)

Exported:
 - sources/osmnx_berlin_vet_clinics_latest.geojson
 - cache/osmnx_berlin_vet_clinics_latest.csv
Shape: (175, 22)


## 5. Quick sanity check of exported files (optional)

We verify that the exported GeoJSON and CSV files exist on disk. This
step is optional but helpful to confirm that the snapshot is available
for `01_vet_clinics_osm_lor_join.ipynb`.

In [7]:
from pathlib import Path

print("GeoJSON exists? ", Path("sources/osmnx_berlin_vet_clinics_latest.geojson").exists())
print("CSV exists?     ", Path("cache/osmnx_berlin_vet_clinics_latest.csv").exists())

GeoJSON exists?  True
CSV exists?      True
